# Autonomous Industrial Energy Market & Telemetry Lakehouse

## 1. Executive Overview & Problem Statement

### Context & Importance
In modern deregulated electricity grids, such as the Australian Energy Market Operator (AEMO) National Electricity Market (NEM), power dispatch values, load demands, and pricing are settled dynamically every **5 minutes**. For high-yield industrial consumers, utility-scale battery operators, and mining conglomerates, real-time exposure to these dispatch intervals dictates immediate financial margins. Operating in this environment presents complex engineering challenges:
- **High Frequency & Velocity**: Large volumes of 5-minute telemetry intervals must be parsed, validated, and processed continuously.
- **Extreme Price Volatility**: Prices routinely spike to extreme regulatory maximums (up to $16,600/MWh) or fall to negative price floors (-$1,000/MWh) during oversupply conditions, making outlier detection tricky.
- **Late-Arriving Data & Revisions**: Retrospective settlement adjustments and system re-runs create duplicate keys and record revisions.

### Architectural Design: The Medallion Pattern
This lakehouse uses the **Medallion Architecture** built on ACID-compliant transaction logs to process raw energy telemetry into highly optimized analytical marts safely and idempotently.

```
  [ AEMO Dispatch Portal (ZIP/CSV) ]
                 │
                 ▼
  ┌──────────────────────────────┐
  │        BRONZE LAYER          │  ◄── Byte-for-byte Raw CSV Archive
  │ (Untouched S3/Blob Storage)  │      Audit Trail (SHA-256 Hashing)
  └──────────────┬───────────────┘
                 │
                 ▼
  ┌──────────────────────────────┐
  │         SILVER LAYER         │  ◄── Strict Data Contract Validation (Pydantic)
  │         (Delta Lake)         │      Idempotent Upsert (MERGE on composite keys)
  └──────┬────────────────┬──────┘
         │                │
         ▼                ▼
  ┌──────────────┐┌──────────────┐
  │  VALID GOLD  ││  QUARANTINE  │  ◄── Non-blocking Quarantine Storage for
  │  (Analytical ││ (Delta Lake) │      corrupted records & contract violations
  │  Star Mart)  │└──────────────┘
  └──────────────┘
```

### Core Engineering Tool Stack
* **Prefect**: Provides resilient, self-healing workflow orchestration with deterministic retries, failure tracking, and pipeline state management.
* **Delta Lake**: Guarantees ACID transactions, point-in-time time travel, schema enforcement, and highly efficient MERGE statements directly in object storage.
* **DuckDB**: Serves as a zero-overhead, highly performant OLAP query engine, enabling immediate, cost-efficient, in-memory SQL queries across your Delta tables.
* **Pydantic**: Enforces strict, declarative type and data contracts at runtime.

## 2. Configuration & Dynamic Thresholds

Hardcoding regulatory thresholds or operational boundaries within ingestion code causes fragile pipeline operations when rules shift (e.g., annual regulatory indexation). To prevent this, we construct a strict, declarative configuration model using **Pydantic** to validate boundaries like NEM pricing constraints and valid regional nodes dynamically.

In [12]:
!pip install -q prefect pydantic duckdb deltalake pyarrow structlog requests pytest ipytest opentelemetry-api opentelemetry-sdk --upgrade

import os
from pydantic import BaseModel, Field

class MarketConfig(BaseModel):
    # Parameterized NEM bounds (indexed annually)
    market_price_floor_aud: float = -1000.00
    market_price_cap_aud: float = 16600.00  # NEM Market Price Cap
    valid_regions: set[str] = {"NSW1", "QLD1", "SA1", "TAS1", "VIC1"}

CONFIG = MarketConfig()

# Clean directory setup
for folder in ["data/bronze", "data/silver", "data/gold", "data/quarantine"]:
    os.makedirs(folder, exist_ok=True)

## 3. Bronze Ingestion & Immutability

The Bronze layer preserves an untouched byte-for-byte history of all source telemetry payloads before transformation. Doing so allows reconstruction of state, replayability of pipeline stages during logic adjustments, and complete audit trace capabilities. Every downloaded asset receives a SHA-256 integrity hash, and a timestamp watermark pattern prevents redundant transfers or telemetry loss during periods of source server offline downtime.

In [2]:
import io
import re
import zipfile
import hashlib
import requests
from datetime import datetime, timezone
from pathlib import Path
import structlog

logger = structlog.get_logger()
AEMO_DISPATCH_URL = "https://www.nemweb.com.au/Reports/CURRENT/DispatchIS_Reports/"

def get_aemo_file_manifest(watermark_ts: str = None) -> list[str]:
    """
    Scans the AEMO portal and returns all dispatch ZIP filenames
    chronologically strictly newer than the watermark.
    """
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(AEMO_DISPATCH_URL, headers=headers, timeout=20)
    resp.raise_for_status()

    matches = re.findall(r'(PUBLIC_DISPATCHIS_(\d{12})_\d+\.zip)', resp.text, re.IGNORECASE)
    # matches: [('PUBLIC_DISPATCHIS_202611031855_...zip', '202611031855'), ...]

    unique_files = sorted(list({m[0]: m[1] for m in matches}.items()), key=lambda x: x[1])

    if not watermark_ts:
        # Default to the single newest file if no watermark exists
        return [unique_files[-1][0]] if unique_files else []

    # Backfill/Incremental: return all files newer than watermark
    return [filename for filename, ts in unique_files if ts > watermark_ts]

def ingest_raw_bronze(filename: str, target_dir: str = "data/bronze") -> dict:
    """
    Downloads file and saves 100% untouched raw CSV bytes.
    Stores metadata separately in an audit record.
    """
    file_url = f"{AEMO_DISPATCH_URL}{filename}"
    resp = requests.get(file_url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        csv_name = z.namelist()[0]
        raw_bytes = z.read(csv_name)

    # Compute SHA-256 for audit immutability verification
    sha256_hash = hashlib.sha256(raw_bytes).hexdigest()

    # Partition by arrival date
    now_utc = datetime.now(timezone.utc)
    out_dir = Path(target_dir) / now_utc.strftime("year=%Y/month=%m/day=%d")
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_path = out_dir / filename.replace(".zip", ".raw.csv")

    # Write byte-for-byte untouched raw file
    with open(raw_path, "wb") as f:
        f.write(raw_bytes)

    audit_metadata = {
        "bronze_path": str(raw_path),
        "source_url": file_url,
        "sha256_hash": sha256_hash,
        "ingested_at_utc": now_utc.isoformat(),
        "file_timestamp": filename.split("_")[2] # e.g. '202611031855'
    }
    return audit_metadata

## 4. Silver Processing & Non-Blocking Quarantine

Production systems must not fail completely when encountering abnormal or malformed payloads. We introduce the **Quarantine Pattern**:
1. **Data Contract Validation**: The schema parses intervals, localizes time signatures from Australian Eastern Standard Time (AEST) to UTC, and verifies bounds.
2. **Split Validation**: Clean records are routed immediately into an idempotent Delta Lake MERGE block.
3. **Granular Quarantine Tagging**: Faulty records are cataloged with specific validation fault codes (e.g. `EXCEEDS_CAP`, `INVALID_REGION`) and appended to a quarantine directory, keeping downstream analytics untainted without crashing the ingestion flow.

In [16]:
import os
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

def validate_and_tag_quarantine(records: list[dict], config: MarketConfig = CONFIG) -> tuple[list[dict], list[dict]]:
    """
    Validates records against market boundaries and records specific failure reasons.
    """
    clean, quarantined = [], []

    for r in records:
        reasons = []
        if not r.get("region_id"):
            reasons.append("MISSING_REGION_KEY")
        elif r["region_id"] not in config.valid_regions:
            reasons.append(f"INVALID_REGION:{r['region_id']}")

        if not r.get("interval_utc"):
            reasons.append("MISSING_INTERVAL_KEY")

        rrp = r.get("rrp_price_aud_per_mwh")
        if rrp is None:
            reasons.append("NULL_PRICE")
        elif rrp < config.market_price_floor_aud:
            reasons.append(f"BELOW_FLOOR:{rrp}<{config.market_price_floor_aud}")
        elif rrp > config.market_price_cap_aud:
            reasons.append(f"EXCEEDS_CAP:{rrp}>{config.market_price_cap_aud}")

        if reasons:
            r["_quarantine_reasons"] = "; ".join(reasons)
            r["_quarantined_at_utc"] = pd.Timestamp.utcnow()
            quarantined.append(r)
        else:
            clean.append(r)

    return clean, quarantined

def process_bronze_to_silver(bronze_path: str, audit_meta: dict, silver_path: str = "data/silver/dispatch_price", quarantine_path: str = "data/quarantine/dispatch_price") -> tuple[int, int]:
    with open(bronze_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f]

    price_lines = [line.split(',') for line in lines if line.startswith("D,DISPATCH,PRICE,")]
    if not price_lines:
        raise ValueError("No price records extracted from Bronze CSV.")

    parsed_records = []
    for parts in price_lines:
        parts = [p.strip().strip('"') for p in parts]
        raw_settlement_date = parts[4]
        region_id = parts[6]
        intervention_flag = int(parts[8]) if parts[8] else 0
        rrp = float(parts[9]) if parts[9] else None
        eep = float(parts[10]) if len(parts) > 10 and parts[10] else None

        # Convert fixed AEST to UTC
        dt_aest = pd.to_datetime(raw_settlement_date, format="%Y/%m/%d %H:%M:%S").tz_localize("Etc/GMT-10")
        interval_utc = dt_aest.tz_convert("UTC")

        parsed_records.append({
            "region_id": region_id,
            "interval_utc": interval_utc,
            "raw_settlement_date": raw_settlement_date,
            "intervention_flag": intervention_flag,
            "rrp_price_aud_per_mwh": rrp,
            "eep_price_aud_per_mwh": eep,
            "_ingested_file": os.path.basename(bronze_path),
            "_ingested_sha256": audit_meta.get("sha256_hash"),
            "_silver_processed_at_utc": pd.Timestamp.utcnow()
        })

    clean_records, quarantined_records = validate_and_tag_quarantine(parsed_records)

    # 1. Write quarantined rows if any
    if quarantined_records:
        arrow_quar = pa.Table.from_pandas(pd.DataFrame(quarantined_records))
        write_deltalake(quarantine_path, arrow_quar, mode="append")

    # 2. Idempotent MERGE for clean rows
    if clean_records:
        arrow_clean = pa.Table.from_pandas(pd.DataFrame(clean_records))
        if not os.path.exists(os.path.join(silver_path, "_delta_log")):
            write_deltalake(silver_path, arrow_clean, mode="error")
        else:
            dt = DeltaTable(silver_path)
            (
                dt.merge(
                    source=arrow_clean,
                    predicate="target.region_id = source.region_id AND target.interval_utc = source.interval_utc",
                    source_alias="source",
                    target_alias="target"
                )
                .when_matched_update_all()
                .when_not_matched_insert_all()
                .execute()
            )

    return len(clean_records), len(quarantined_records)

## 5. Gold Analytical Marts & FinOps Housekeeping

### Hourly Analytical Rollup
The Gold layer aggregates 5-minute granular intervals into hourly analytical summaries (min, max, and volume-weighted average clearing rates). We execute this OLAP transformation directly over parquet-backed Delta tables using DuckDB.

### Managing the "Small File Problem"
High-frequency streaming and micro-batch pipelines generate millions of tiny parquet files over time, severely degrading downstream scan performance and inflating storage API call costs. To address this, we trigger an `OPTIMIZE` instruction to compact adjacent small files into consolidated, bin-packed parquet files.

In [4]:
import duckdb
from deltalake import DeltaTable, write_deltalake
import pyarrow as pa

def process_silver_to_gold(silver_path: str = "data/silver/dispatch_price", gold_path: str = "data/gold/fact_hourly_regional_prices") -> int:
    con = duckdb.connect()
    query = f"""
        SELECT
            date_trunc('hour', interval_utc) AS hour_interval_utc,
            region_id,
            count(*) AS interval_count,
            round(avg(rrp_price_aud_per_mwh), 2) AS avg_rrp_aud_per_mwh,
            round(min(rrp_price_aud_per_mwh), 2) AS min_rrp_aud_per_mwh,
            round(max(rrp_price_aud_per_mwh), 2) AS max_rrp_aud_per_mwh,
            max(intervention_flag) AS had_intervention,
            current_timestamp AS _gold_processed_at_utc
        FROM delta_scan('{silver_path}')
        GROUP BY 1, 2
        ORDER BY 1 DESC, 2 ASC
    """
    gold_df = con.execute(query).df()
    arrow_gold = pa.Table.from_pandas(gold_df)
    write_deltalake(gold_path, arrow_gold, mode="overwrite")
    return len(gold_df)

def optimize_silver(silver_path: str = "data/silver/dispatch_price"):
    dt = DeltaTable(silver_path)
    return dt.optimize.compact()

## 6. Pipeline Orchestration with Prefect

Ad-hoc scheduler scripts lack built-in status monitoring, conditional node execution, and self-healing mechanisms. Here, we encapsulate our ingestion, validation, aggregation, and cleanup operations into a cohesive **Prefect Flow**, utilizing retries, state handshakes, and strict parameter checking.

In [23]:
import os
# Disable Prefect's OpenTelemetry tracking to resolve version conflicts with Colab defaults
os.environ["PREFECT_INSTRUMENTATION_OPENTELEMETRY_ENABLED"] = "False"

from prefect import task, flow
from typing import Optional

@task(retries=3, retry_delay_seconds=5)
def task_ingest(filename: str):
    return ingest_raw_bronze(filename)

@task
def task_silver(audit_meta: dict):
    return process_bronze_to_silver(audit_meta["bronze_path"], audit_meta)

@task
def task_gold():
    return process_silver_to_gold()

@task
def task_finops():
    return optimize_silver()

@flow(name="aemo_medallion_lakehouse_flow")
def run_lakehouse_pipeline(watermark_ts: Optional[str] = None):
    # 1. Discover files past watermark
    files_to_process = get_aemo_file_manifest(watermark_ts)
    print(f"Found {len(files_to_process)} file(s) to process.")

    last_watermark = watermark_ts
    for fname in files_to_process:
        meta = task_ingest(fname)
        clean_cnt, quar_cnt = task_silver(meta)
        print(f"[{fname}] Silver processed: {clean_cnt} clean, {quar_cnt} quarantined.")
        last_watermark = meta["file_timestamp"]

    # 2. Update downstream marts
    gold_cnt = task_gold()
    task_finops()

    return {
        "processed_files": len(files_to_process),
        "new_watermark": last_watermark,
        "gold_rows": gold_cnt
    }

# Execute flow
run_result = run_lakehouse_pipeline()
print("Pipeline Run:", run_result)

INFO:prefect.flow_runs:Beginning flow run 'mauve-jellyfish' for flow 'aemo_medallion_lakehouse_flow'


Found 1 file(s) to process.


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


[PUBLIC_DISPATCHIS_202611031950_0000000546464885.zip] Silver processed: 5 clean, 0 quarantined.


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.flow_runs:Finished in state Completed()


Pipeline Run: {'processed_files': 1, 'new_watermark': '202611031950', 'gold_rows': 5}


## 7. Automated Verification & Pytest Suite

Reliability is verified through automated unit tests. Below, we test:
1. **Granular Quarantine Routing**: Verifies contract checks correctly catch violations and log the exact structural error code.
2. **Silver Merge Idempotency**: Running identical files against Delta multiple times must *never* result in duplicate target records.

In [24]:
import pytest
import pandas as pd
from deltalake import DeltaTable

def test_quarantine_granular_reasons():
    bad_batch = [
        {"region_id": "INVALID_STATE", "interval_utc": pd.Timestamp.utcnow(), "rrp_price_aud_per_mwh": 100.0},
        {"region_id": "NSW1", "interval_utc": pd.Timestamp.utcnow(), "rrp_price_aud_per_mwh": 50000.0},
        {"region_id": "VIC1", "interval_utc": pd.Timestamp.utcnow(), "rrp_price_aud_per_mwh": -1500.0}
    ]
    clean, quar = validate_and_tag_quarantine(bad_batch)
    assert len(clean) == 0
    assert len(quar) == 3
    assert "INVALID_REGION" in quar[0]["_quarantine_reasons"]
    assert "EXCEEDS_CAP" in quar[1]["_quarantine_reasons"]
    assert "BELOW_FLOOR" in quar[2]["_quarantine_reasons"]

def test_silver_merge_idempotency(tmp_path):
    """
    Running the exact same batch twice MUST yield identical row counts in Delta.
    """
    test_silver = str(tmp_path / "test_silver")
    test_meta = {"sha256_hash": "mock_hash"}

    # Mock raw CSV
    csv_content = (
        "C,NEM,DISPATCH,PRICE,1\n"
        "D,DISPATCH,PRICE,1,\"2026/11/03 19:00:00\",1,NSW1,1,0,120.50,0.0\n"
        "D,DISPATCH,PRICE,1,\"2026/11/03 19:00:00\",1,VIC1,1,0,140.20,0.0\n"
    )
    raw_file = tmp_path / "mock.raw.csv"
    raw_file.write_text(csv_content)

    # First Run
    clean1, _ = process_bronze_to_silver(str(raw_file), test_meta, silver_path=test_silver, quarantine_path=str(tmp_path / "q"))
    dt = DeltaTable(test_silver)
    count_after_first = dt.to_pyarrow_table().num_rows

    # Second Run (Duplicate)
    clean2, _ = process_bronze_to_silver(str(raw_file), test_meta, silver_path=test_silver, quarantine_path=str(tmp_path / "q"))
    dt_reloaded = DeltaTable(test_silver)
    count_after_second = dt_reloaded.to_pyarrow_table().num_rows

    assert count_after_first == 2
    assert count_after_second == 2  # Idempotency confirmed: No duplicate rows created!

# Run in-notebook pytest
import ipytest
ipytest.autoconfig()
ipytest.run()

..                                                                                           [100%]
========================================= warnings summary =========================================
../usr/local/lib/python3.13/dist-packages/_pytest/config/__init__.py:1507
  /usr/local/lib/python3.13/dist-packages/_pytest/config/__init__.py:1507: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
2 passed, 1 warning in 0.08s


<ExitCode.OK: 0>

## 8. Final Verification & Data Inspection

With our Prefect pipeline successfully executed and unit tests verified, we will query our newly generated Bronze, Silver, Gold, and Quarantine datasets. We leverage DuckDB to run low-latency SQL directly over the Delta files to verify pipeline data flow.

In [25]:
import duckdb
import os

con = duckdb.connect()

print("=== INSPECTING SILVER DISPATCH TABLE (Delta Lake) ===")
try:
    # Dynamic scan of available fields in the Delta path
    con.sql("SELECT * FROM delta_scan('data/silver/dispatch_price') LIMIT 5").show()
except Exception as e:
    print(f"Could not query Silver Delta: {e}")

print("\n=== INSPECTING GOLD REGIONAL MART (Delta Lake) ===")
try:
    con.sql("SELECT * FROM delta_scan('data/gold/fact_hourly_regional_prices') LIMIT 5").show()
except Exception as e:
    print(f"Could not query Gold Delta: {e}")

if os.path.exists("data/quarantine/dispatch_price"):
    print("\n=== INSPECTING QUARANTINE DATA (Delta Lake) ===")
    try:
        con.sql("SELECT * FROM delta_scan('data/quarantine/dispatch_price') LIMIT 5").show()
    except Exception as e:
        print(f"Could not query Quarantine Delta: {e}")
else:
    print("\nNo quarantined records in current run (all ingested data met the schema contract).")

=== INSPECTING SILVER DISPATCH TABLE (Delta Lake) ===
┌───────────┬──────────────────────────┬─────────────────────┬───────────────────┬───────────────────────┬───────────────────────┬─────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────┬───────────────────────────────┐
│ region_id │       interval_utc       │ raw_settlement_date │ intervention_flag │ rrp_price_aud_per_mwh │ eep_price_aud_per_mwh │                     _ingested_file                      │                         _ingested_sha256                         │   _silver_processed_at_utc    │
│  varchar  │ timestamp with time zone │       varchar       │       int64       │        double         │        double         │                         varchar                         │                             varchar                              │   timestamp with time zone    │
├───────────┼──────────────────────────┼─────────────────────┼───────────────────┼